# S45_03 — HuggingFace Datasets Library

The `datasets` library provides a unified interface to 50k+ public datasets with memory-mapped Arrow storage — datasets larger than RAM work without loading everything into memory.

## Loading datasets

In [ ]:
# pip install datasets
from datasets import load_dataset

# Load from HuggingFace Hub
ds = load_dataset('imdb')           # sentiment — 25k train, 25k test
print(ds)
# DatasetDict({'train': Dataset({features: ['text', 'label'], num_rows: 25000}), ...})

# Access a split
train = ds['train']
print(train[0])                     # first example as dict
print(train['label'][:5])           # slice a column

In [ ]:
# Load a specific config or subset
glue_sst = load_dataset('glue', 'sst2')   # config = task name
print(glue_sst)

# Load only a fraction (useful for prototyping)
small = load_dataset('imdb', split='train[:1%]')   # 250 examples
print(f'Small split: {len(small)} examples')

In [ ]:
# Load from local files
from datasets import Dataset
import pandas as pd

# From a pandas DataFrame
df = pd.DataFrame({'text': ['Good film', 'Terrible acting', 'Loved it'], 'label': [1, 0, 1]})
local_ds = Dataset.from_pandas(df)
print(local_ds)

# From CSV / JSON
# csv_ds = load_dataset('csv', data_files='data.csv')
# json_ds = load_dataset('json', data_files='data.jsonl')

## Mapping and transformation

In [ ]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained('bert-base-uncased')

def tokenize(examples):
    # examples is a batch dict: {'text': [...], 'label': [...]}
    return tokenizer(examples['text'], truncation=True, max_length=128)

# map() applies function to every batch — returns a new Dataset
tokenized = train.map(tokenize, batched=True, batch_size=1000)
print(tokenized.column_names)   # adds input_ids, attention_mask, token_type_ids

# Remove columns you no longer need
tokenized = tokenized.remove_columns(['text'])
tokenized = tokenized.rename_column('label', 'labels')  # PyTorch convention
tokenized.set_format('torch')   # return torch.Tensors from __getitem__
print(tokenized[0])

In [ ]:
# Filtering
long_reviews = train.filter(lambda x: len(x['text'].split()) > 200)
print(f'Long reviews: {len(long_reviews):,}')

# Shuffle and select
sample = train.shuffle(seed=42).select(range(500))
print(f'Sample size: {len(sample)}')

# Train/test split (for datasets with only a train split)
splits = sample.train_test_split(test_size=0.2, seed=42)
print(splits)

## Streaming — datasets larger than RAM

In [ ]:
# streaming=True returns an IterableDataset — no download, reads on-the-fly
streamed = load_dataset('c4', 'en', split='train', streaming=True, trust_remote_code=True)

# IterableDataset supports map, filter, shuffle (buffer-based), take
sample_iter = streamed.take(5)   # first 5 examples
for example in sample_iter:
    print(example['text'][:100])
    print('---')

# Shuffle a streaming dataset (buffer shuffling)
shuffled_iter = streamed.shuffle(seed=42, buffer_size=1000)

## DataCollator — dynamic batching for training

In [ ]:
from transformers import DataCollatorWithPadding
from torch.utils.data import DataLoader

# DataCollatorWithPadding pads each batch to the longest sequence IN that batch
# This is more memory-efficient than padding to global max_length at preprocessing time
collator = DataCollatorWithPadding(tokenizer=tokenizer)

loader = DataLoader(
    tokenized.select(range(100)),  # small slice for demo
    batch_size=16,
    shuffle=True,
    collate_fn=collator,
)

batch = next(iter(loader))
print({k: v.shape for k, v in batch.items()})
# input_ids: (16, <variable padded length>)

## Saving and loading processed datasets

In [ ]:
# Save to disk (Arrow format — fast reload, memory-mappable)
# tokenized.save_to_disk('./tokenized_imdb')

# Load back
# from datasets import load_from_disk
# tokenized = load_from_disk('./tokenized_imdb')

# Push to Hub
# tokenized.push_to_hub('my-username/imdb-bert-tokenized')

print('Key workflow:')
print('1. load_dataset()  → raw')
print('2. .map(tokenize, batched=True)  → tokenized')
print('3. .save_to_disk() / load_from_disk()  → reuse without re-processing')
print('4. DataLoader + DataCollatorWithPadding  → training')

Next: [S45_04_peft_lora.ipynb](./S45_04_peft_lora.ipynb)